# Revenue Leakage & Cohort Churn Analysis

## Notebook Objective

Clean the raw transaction dataset while preserving legitimate business events such as returns and cancellations.

## Cleaning Objectives

- Remove true duplicate rows
- Handle missing values appropriately
- Remove accounting adjustment entries
- Preserve valid return transactions
- Prepare a clean dataset for exploratory analysis

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [2]:
# Load the original raw dataset
file_path = "../data/raw/online_retail_II.xlsx"

df = pd.read_excel(file_path)

print(f"Dataset loaded successfully: {df.shape}")

Dataset loaded successfully: (525461, 8)


In [3]:

print(f"Dataset loaded successfully: {df.shape}")

Dataset loaded successfully: (525461, 8)


In [4]:
# Store the original number of rows
original_rows = len(df)

# Remove duplicate rows
df = df.drop_duplicates()

# Store the new number of rows
cleaned_rows = len(df)

# Calculate how many rows were removed
duplicates_removed = original_rows - cleaned_rows

print(f"Original Rows      : {original_rows:,}")
print(f"Rows After Cleaning: {cleaned_rows:,}")
print(f"Duplicates Removed : {duplicates_removed:,}")

Original Rows      : 525,461
Rows After Cleaning: 518,596
Duplicates Removed : 6,865


In [5]:
# Verify that no duplicate rows remain
remaining_duplicates = df.duplicated().sum()

print(f"Remaining duplicate rows: {remaining_duplicates}")

Remaining duplicate rows: 0


In [6]:
df = df.drop_duplicates(ignore_index=True)

In [7]:
# Business Check:
# Accounting adjustments are not retail sales.
# We identify them before removing them from the transactional dataset.

adjustments = df[df["Description"] == "Adjust bad debt"]

print(f"Accounting adjustment records: {len(adjustments)}")

adjustments

Accounting adjustment records: 3


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
177338,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
273025,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
398731,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


In [8]:
# Remove accounting adjustment entries
df = df[df["Description"] != "Adjust bad debt"].reset_index(drop=True)

print(f"Dataset shape after removing accounting adjustments: {df.shape}")

Dataset shape after removing accounting adjustments: (518593, 8)


In [9]:
# Verify that no accounting adjustment records remain
remaining_adjustments = (
    df["Description"] == "Adjust bad debt"
).sum()

print(f"Remaining accounting adjustment records: {remaining_adjustments}")

Remaining accounting adjustment records: 0


In [10]:
# Investigate transactions with missing product descriptions
missing_description = df[df["Description"].isna()]

print(f"Transactions with missing descriptions: {len(missing_description)}")

missing_description.head(10)

Transactions with missing descriptions: 2928


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
462,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3077,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3124,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3687,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4233,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4499,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6299,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6476,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6497,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6502,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


In [11]:
# Remove rows with missing product descriptions
original_rows = len(df)

df = df.dropna(subset=["Description"]).reset_index(drop=True)

cleaned_rows = len(df)

rows_removed = original_rows - cleaned_rows

print(f"Original Rows      : {original_rows:,}")
print(f"Rows After Cleaning: {cleaned_rows:,}")
print(f"Rows Removed       : {rows_removed:,}")

Original Rows      : 518,593
Rows After Cleaning: 515,665
Rows Removed       : 2,928


In [12]:
remaining_missing_descriptions = df["Description"].isna().sum()

print(f"Remaining missing descriptions: {remaining_missing_descriptions}")

Remaining missing descriptions: 0


# Data Cleaning Summary

## Cleaning Steps Performed

| Step | Rows Removed | Reason |
|------|-------------:|--------|
| Removed duplicate records | 6,865 | Prevent double counting of revenue and transactions |
| Removed accounting adjustments | 3 | Financial ledger entries, not customer purchases |
| Removed records with missing descriptions | 2,928 | Missing product information and zero-priced system entries |

## Dataset Summary

- Original Rows: **525,461**
- Final Rows: **515,665**
- Total Rows Removed: **9,796**

## Business Decisions

- Preserved return transactions (negative quantities)
- Preserved transactions with missing Customer IDs for revenue analysis
- Will create a separate dataset for cohort analysis later